# 🔐 CIFRADO ELGAMAL

**Laboratorio de Criptografía Asimétrica**

---

## Estructura del Notebook

1. **PARTE PRÁCTICA** - Ejecuta y experimenta con ElGamal
2. **PARTE TEÓRICA** - Fundamentos matemáticos y explicación detallada

---

**ElGamal** es un sistema de criptografía asimétrica (clave pública) basado en el problema del logaritmo discreto.

**Componentes:**
- **Clave pública:** (p, g, h) donde p=primo, g=generador, h=g^x mod p
- **Clave privada:** x (número secreto)

**Fórmulas:**
- **Cifrado:** (c1, c2) = (g^k mod p, m·h^k mod p)
- **Descifrado:** m = c2·(c1^x)^(-1) mod p

**Alfabeto:** A=0, B=1, C=2, ..., Z=25

---

# 📝 PARTE PRÁCTICA

## 🎯 Ejemplo Completo: Cifrar y Descifrar "HELLO"

### ⚙️ Configuración del Ejemplo

Para fines educativos, usaremos valores pequeños y conocidos:

In [ ]:
import random

# ============================================
# CONFIGURACIÓN - Valores educativos
# ============================================

# Parámetros públicos del sistema
p = 23          # Primo pequeño (en producción se usan primos de 2048+ bits)
g = 5           # Generador del grupo multiplicativo mod p

# Clave privada (mantenida en secreto)
x = 6           # Número aleatorio entre 1 y p-1

# Clave pública (calculada a partir de la privada)
h = pow(g, x, p)  # h = g^x mod p

# Mensaje a cifrar
MENSAJE = "HELLO"

print("🔑 Claves ElGamal:")
print(f"   Clave pública:  (p={p}, g={g}, h={h})")
print(f"   Clave privada:  x={x}")
print(f"\n📝 Mensaje: {MENSAJE}")
print(f"\n💡 Cálculo de h: {g}^{x} mod {p} = {h}")

### 1. Funciones de Conversión Texto ↔ Números

In [ ]:
def texto_a_numeros(texto):
    """
    Convierte texto a lista de números usando el mapeo A=0, B=1, ..., Z=25.
    
    Proceso:
    1. Limpia el texto (solo letras mayúsculas)
    2. Convierte cada letra: ord(letra) - ord('A')
    3. Retorna lista de números
    
    Ejemplo: "HELLO" → [7, 4, 11, 11, 14]
    """
    texto_limpio = ''.join(c.upper() for c in texto if c.isalpha())
    return [ord(c) - ord('A') for c in texto_limpio]

def numeros_a_texto(numeros):
    """
    Convierte lista de números a texto usando el mapeo 0=A, 1=B, ..., 25=Z.
    
    Proceso:
    1. Para cada número n: chr(n + ord('A'))
    2. Une todos los caracteres
    
    Ejemplo: [7, 4, 11, 11, 14] → "HELLO"
    """
    return ''.join(chr(n + ord('A')) for n in numeros)

# Convertir mensaje a números
numeros_mensaje = texto_a_numeros(MENSAJE)
print(f"Mensaje: {MENSAJE}")
print(f"Números: {numeros_mensaje}")
print(f"\nDetalle:")
for letra, num in zip(MENSAJE, numeros_mensaje):
    print(f"  {letra} → {num:2d}")

### 2. Función: Cifrar un Número con ElGamal

In [ ]:
def cifrar_elgamal(m, p, g, h):
    """
    Cifra un número usando ElGamal.
    
    Proceso paso a paso:
    1. Elegir k aleatorio (efímero) donde 1 < k < p-1
    2. Calcular c1 = g^k mod p
    3. Calcular secreto compartido: s = h^k mod p
    4. Calcular c2 = (m × s) mod p
    5. El cifrado es el par (c1, c2)
    
    Parámetros:
        m: número del mensaje (0-25 para letras)
        p: primo del sistema
        g: generador
        h: clave pública (h = g^x mod p)
    
    Retorna:
        Diccionario con: (c1, c2), k, s y todos los cálculos
    """
    # Validar que m < p
    if m >= p:
        raise ValueError(f"Mensaje ({m}) debe ser menor que p ({p})")
    
    # 1. Elegir k aleatorio (efímero)
    # Nota: en cada cifrado se usa un k diferente (cifrado probabilístico)
    k = random.randrange(2, p - 1)
    
    # 2. Calcular c1 = g^k mod p
    c1 = pow(g, k, p)
    
    # 3. Calcular secreto compartido s = h^k mod p
    s = pow(h, k, p)
    
    # 4. Calcular c2 = (m × s) mod p
    c2 = (m * s) % p
    
    # Retornar todos los detalles para análisis
    return {
        'mensaje': m,
        'k': k,
        'c1': c1,
        'c1_calculo': f"{g}^{k} mod {p} = {c1}",
        's': s,
        's_calculo': f"{h}^{k} mod {p} = {s}",
        'c2': c2,
        'c2_calculo': f"({m} × {s}) mod {p} = {c2}",
        'cifrado': (c1, c2)
    }

# Ejemplo: cifrar la letra 'H' (H=7)
print("Ejemplo: Cifrar 'H' (7)")
print("="*60)
resultado = cifrar_elgamal(7, p, g, h)
print(f"Mensaje: {resultado['mensaje']} (H)")
print(f"k aleatorio: {resultado['k']}")
print(f"c1 = {resultado['c1_calculo']}")
print(f"s = {resultado['s_calculo']}")
print(f"c2 = {resultado['c2_calculo']}")
print(f"\n🔒 Cifrado: ({resultado['c1']}, {resultado['c2']})")

### 3. Función: Descifrar un Mensaje ElGamal

In [ ]:
def descifrar_elgamal(c1, c2, p, x):
    """
    Descifra un mensaje ElGamal usando la clave privada.
    
    Proceso paso a paso:
    1. Calcular secreto compartido: s = c1^x mod p
    2. Calcular inverso modular: s^(-1) mod p
    3. Recuperar mensaje: m = (c2 × s^(-1)) mod p
    
    El descifrador puede calcular s porque conoce x (clave privada):
      s = c1^x mod p = (g^k)^x mod p = g^(kx) mod p
    
    Y esto es igual al s usado en el cifrado:
      s = h^k mod p = (g^x)^k mod p = g^(kx) mod p
    
    Parámetros:
        c1, c2: mensaje cifrado
        p: primo del sistema
        x: clave privada
    
    Retorna:
        Diccionario con: mensaje, s, s_inv y todos los cálculos
    """
    # 1. Calcular secreto compartido s = c1^x mod p
    s = pow(c1, x, p)
    
    # 2. Calcular inverso modular de s usando Pequeño Teorema de Fermat
    # Para p primo: s^(-1) = s^(p-2) mod p
    s_inv = pow(s, p - 2, p)
    
    # 3. Recuperar mensaje m = (c2 × s^(-1)) mod p
    m = (c2 * s_inv) % p
    
    # Retornar todos los detalles
    return {
        'c1': c1,
        'c2': c2,
        's': s,
        's_calculo': f"{c1}^{x} mod {p} = {s}",
        's_inv': s_inv,
        's_inv_calculo': f"{s}^-1 mod {p} = {s_inv}",
        'mensaje': m,
        'm_calculo': f"({c2} × {s_inv}) mod {p} = {m}"
    }

# Ejemplo: descifrar el resultado anterior
print("Ejemplo: Descifrar el mensaje anterior")
print("="*60)
desc_resultado = descifrar_elgamal(resultado['c1'], resultado['c2'], p, x)
print(f"Cifrado: ({desc_resultado['c1']}, {desc_resultado['c2']})")
print(f"s = {desc_resultado['s_calculo']}")
print(f"s⁻¹ = {desc_resultado['s_inv_calculo']}")
print(f"m = {desc_resultado['m_calculo']}")
print(f"\n🔓 Descifrado: {desc_resultado['mensaje']} (H)")
print(f"✓ Correcto: {desc_resultado['mensaje'] == 7}")

### 4. Función: Cifrar Texto Completo

In [ ]:
def cifrar_texto(texto, p, g, h):
    """
    Cifra un texto completo letra por letra.
    
    Cada letra se cifra independientemente con un k diferente,
    lo que hace el cifrado probabilístico (mismo texto → diferentes cifrados).
    
    Parámetros:
        texto: mensaje en texto
        p, g, h: parámetros de la clave pública
    
    Retorna:
        Lista de diccionarios con detalles de cada letra cifrada
    """
    numeros = texto_a_numeros(texto)
    resultados = []
    
    for i, m in enumerate(numeros):
        # Cifrar cada número
        resultado = cifrar_elgamal(m, p, g, h)
        resultado['posicion'] = i
        resultado['letra'] = chr(m + ord('A'))
        resultados.append(resultado)
    
    return resultados

# CIFRAR EL MENSAJE COMPLETO
resultados_cifrado = cifrar_texto(MENSAJE, p, g, h)

print("="*100)
print("PROCESO DE CIFRADO - MENSAJE COMPLETO")
print("="*100)
print(f"\nMensaje original: {MENSAJE}")
print(f"Clave pública: (p={p}, g={g}, h={h})")
print("\n" + "-"*100)
print(f"{'Pos':<4} {'Letra':<6} {'m':<4} {'k':<4} {'c1':<6} {'s':<6} {'c2':<6} {'Cifrado':<15}")
print("-"*100)

for r in resultados_cifrado:
    print(f"{r['posicion']:<4} {r['letra']:<6} {r['mensaje']:<4} {r['k']:<4} "
          f"{r['c1']:<6} {r['s']:<6} {r['c2']:<6} ({r['c1']}, {r['c2']})")

print("\n" + "="*100)
print("🔒 MENSAJE CIFRADO:")
for r in resultados_cifrado:
    print(f"   {r['letra']}: ({r['c1']}, {r['c2']})")
print("="*100)

print("\n💡 Nota: Cada letra usa un k diferente (cifrado probabilístico)")

### 5. Función: Descifrar Texto Completo

In [ ]:
def descifrar_texto(mensajes_cifrados, p, x):
    """
    Descifra una lista de mensajes cifrados.
    
    Parámetros:
        mensajes_cifrados: lista de diccionarios con cifrados
        p: primo del sistema
        x: clave privada
    
    Retorna:
        Tupla (texto_descifrado, lista_de_detalles)
    """
    resultados = []
    numeros_descifrados = []
    
    for msg in mensajes_cifrados:
        c1, c2 = msg['cifrado']
        resultado = descifrar_elgamal(c1, c2, p, x)
        resultado['posicion'] = msg['posicion']
        resultado['letra_original'] = msg['letra']
        resultado['letra_descifrada'] = chr(resultado['mensaje'] + ord('A'))
        resultados.append(resultado)
        numeros_descifrados.append(resultado['mensaje'])
    
    texto_descifrado = numeros_a_texto(numeros_descifrados)
    return texto_descifrado, resultados

# DESCIFRAR EL MENSAJE
texto_descifrado, resultados_descifrado = descifrar_texto(resultados_cifrado, p, x)

print("="*100)
print("PROCESO DE DESCIFRADO")
print("="*100)
print(f"\nClave privada: x={x}")
print("\n" + "-"*100)
print(f"{'Pos':<4} {'Cifrado':<15} {'s':<6} {'s⁻¹':<6} {'m':<4} {'Letra':<6}")
print("-"*100)

for r in resultados_descifrado:
    print(f"{r['posicion']:<4} ({r['c1']}, {r['c2']}){' ':<6} {r['s']:<6} "
          f"{r['s_inv']:<6} {r['mensaje']:<4} {r['letra_descifrada']:<6}")

print("\n" + "="*100)
print(f"🔓 MENSAJE DESCIFRADO: {texto_descifrado}")
print("="*100)

# Verificación
print(f"\n✅ Verificación: {MENSAJE} == {texto_descifrado} → {MENSAJE == texto_descifrado}")

### 📊 Resumen del Ejemplo

In [ ]:
print("="*100)
print(" "*40 + "RESUMEN")
print("="*100)
print(f"\nMensaje original:    {MENSAJE}")
print(f"\nClaves:")
print(f"  • Pública:  (p={p}, g={g}, h={h})")
print(f"  • Privada:  x={x}")
print(f"\nCifrado:")
for r in resultados_cifrado:
    print(f"  {r['letra']}: ({r['c1']}, {r['c2']})")
print(f"\nMensaje descifrado:  {texto_descifrado}")
print(f"\n✓ Cifrado y descifrado exitosos!")
print("="*100)

---

## 🧪 Casos de Uso para Probar

### Caso 1: Mensaje Corto

In [ ]:
print("CASO 1: HI")
print("-"*60)
mensaje1 = "HI"
cifrado1 = cifrar_texto(mensaje1, p, g, h)
descifrado1, _ = descifrar_texto(cifrado1, p, x)
print(f"Original:   {mensaje1}")
print(f"Cifrado:    {[(r['c1'], r['c2']) for r in cifrado1]}")
print(f"Descifrado: {descifrado1}")
print(f"✓ Correcto: {mensaje1 == descifrado1}\n")

### Caso 2: Palabra Completa

In [ ]:
print("CASO 2: CRYPTO")
print("-"*60)
mensaje2 = "CRYPTO"
cifrado2 = cifrar_texto(mensaje2, p, g, h)
descifrado2, _ = descifrar_texto(cifrado2, p, x)
print(f"Original:   {mensaje2}")
print(f"Descifrado: {descifrado2}")
print(f"✓ Correcto: {mensaje2 == descifrado2}\n")

### Caso 3: Cifrado Probabilístico

In [ ]:
print("CASO 3: CIFRADO PROBABILÍSTICO - Mismo mensaje, diferentes cifrados")
print("-"*80)
mensaje3 = "AAA"
print(f"Mensaje: {mensaje3} (misma letra 3 veces)\n")

cifrado3_1 = cifrar_texto(mensaje3, p, g, h)
cifrado3_2 = cifrar_texto(mensaje3, p, g, h)

print("Primera cifrado:")
for r in cifrado3_1:
    print(f"  A: ({r['c1']}, {r['c2']})")

print("\nSegundo cifrado (mismo mensaje):")
for r in cifrado3_2:
    print(f"  A: ({r['c1']}, {r['c2']})")

print("\n💡 Los cifrados son diferentes porque se usa un k aleatorio diferente cada vez.")
print("   Esto es una característica de seguridad importante de ElGamal.")

---

# 📚 PARTE TEÓRICA

## 1. ¿Qué es el Cifrado ElGamal?

El **Cifrado ElGamal** es un sistema de criptografía asimétrica propuesto por Taher ElGamal en 1985. Se basa en el **problema del logaritmo discreto**, considerado computacionalmente difícil.

### Características Principales

1. **Criptografía Asimétrica**: Usa par de claves pública/privada
2. **Cifrado Probabilístico**: Mismo mensaje → diferentes cifrados
3. **Seguridad Semántica**: Atacante no puede obtener información del cifrado
4. **Expansión del Mensaje**: Cifrado es 2× el tamaño del mensaje

### Componentes del Sistema

**Parámetros Públicos:**
- `p`: Número primo grande (en producción: 2048+ bits)
- `g`: Generador del grupo multiplicativo módulo p
- `h`: Parte de la clave pública, calculada como h = g^x mod p

**Claves:**
- **Pública**: (p, g, h) - Se comparte libremente
- **Privada**: x - Se mantiene en secreto

### Fórmulas Matemáticas

**Generación de Claves:**
```
1. Elegir primo grande p
2. Elegir generador g del grupo Z*p
3. Elegir clave privada x (1 < x < p-1)
4. Calcular h = g^x mod p
```

**Cifrado de mensaje m:**
```
1. Elegir k aleatorio (1 < k < p-1)
2. c1 = g^k mod p
3. c2 = m·h^k mod p
4. Cifrado = (c1, c2)
```

**Descifrado de (c1, c2):**
```
1. s = c1^x mod p
2. m = c2·s^(-1) mod p
```

### ¿Por qué funciona?

El descifrador puede recuperar el mensaje porque:

```
s = c1^x mod p = (g^k)^x mod p = g^(kx) mod p
```

Y en el cifrado se usó:

```
h^k = (g^x)^k mod p = g^(kx) mod p
```

Por lo tanto:

```
m = c2·s^(-1) = (m·h^k)·(h^k)^(-1) = m
```

## 2. Problema del Logaritmo Discreto

### Definición

Dado:
- Un primo p
- Un generador g
- Un valor h = g^x mod p

**Problema:** Encontrar x

Este problema es considerado **computacionalmente difícil** para primos grandes (no existe algoritmo eficiente conocido).

### Ejemplo Pequeño

```
p = 23, g = 5
h = 8
```

¿Cuál es x tal que 5^x mod 23 = 8?

Probando:
```
5^1 mod 23 = 5
5^2 mod 23 = 2
5^3 mod 23 = 10
5^4 mod 23 = 4
5^5 mod 23 = 20
5^6 mod 23 = 8  ✓
```

Respuesta: x = 6

Para números pequeños es fácil por fuerza bruta, pero para primos de 2048 bits es prácticamente imposible.

In [ ]:
# Demostración: tabla de logaritmos discretos para p=23, g=5
print("Logaritmos Discretos: 5^x mod 23")
print("="*40)
print(f"{'x':<5} {'5^x mod 23':<15}")
print("-"*40)

for x_val in range(1, 23):
    resultado = pow(5, x_val, 23)
    print(f"{x_val:<5} {resultado:<15}")

print("\n💡 Para h=8, encontramos x=6")

## 3. Generadores del Grupo Multiplicativo

### ¿Qué es un Generador?

Un generador `g` del grupo multiplicativo módulo `p` es un número tal que:

```
{g^1 mod p, g^2 mod p, ..., g^(p-1) mod p}
```

genera todos los números de 1 a p-1.

### Ejemplo: ¿Es 5 un generador mod 23?

Calculemos todas las potencias de 5 mod 23:

In [ ]:
p_ejemplo = 23
g_ejemplo = 5

potencias = set()
print(f"Potencias de {g_ejemplo} mod {p_ejemplo}:")
print("="*50)

for i in range(1, p_ejemplo):
    valor = pow(g_ejemplo, i, p_ejemplo)
    potencias.add(valor)
    if i <= 10 or i > p_ejemplo - 3:
        print(f"{g_ejemplo}^{i:2d} mod {p_ejemplo} = {valor:2d}")
    elif i == 11:
        print("   ...")

print(f"\nNúmeros generados: {len(potencias)} de {p_ejemplo-1} posibles")
print(f"¿Es generador? {len(potencias) == p_ejemplo - 1}")

if len(potencias) == p_ejemplo - 1:
    print(f"\n✓ {g_ejemplo} es un generador de Z*{p_ejemplo}")

## 4. Cifrado Probabilístico

### ¿Qué significa "probabilístico"?

En ElGamal, el mismo mensaje cifrado múltiples veces produce **cifrados diferentes** cada vez.

Esto ocurre porque se elige un **k aleatorio** diferente en cada cifrado.

### Ventaja de Seguridad

Un atacante que intercepta dos cifrados del mismo mensaje no puede determinar que son el mismo mensaje, porque los cifrados son completamente diferentes.

### Ejemplo: Cifrar "A" tres veces

In [ ]:
print("Cifrando 'A' (0) tres veces con la misma clave:")
print("="*60)

for i in range(1, 4):
    resultado = cifrar_elgamal(0, p, g, h)
    print(f"\nCifrado {i}:")
    print(f"  k = {resultado['k']}")
    print(f"  Cifrado = ({resultado['c1']}, {resultado['c2']})")

print("\n💡 Mismo mensaje (A), pero cifrados completamente diferentes.")
print("   Esto hace imposible el análisis de frecuencias.")

## 5. Seguridad de ElGamal

### Fundamento de Seguridad

La seguridad de ElGamal se basa en dos problemas difíciles:

1. **Problema del Logaritmo Discreto (DLP)**
   - Encontrar x dado g^x mod p = h

2. **Problema Computational Diffie-Hellman (CDH)**
   - Calcular g^(xy) mod p dados g^x y g^y

### Parámetros Recomendados (2024)

| Nivel de Seguridad | Tamaño de p (bits) |
|-------------------|-----------------|
| Bajo (educación)  | 1024            |
| Medio             | 2048            |
| Alto              | 3072+           |

### Ventajas

✓ Seguridad semántica (cifrado probabilístico)
✓ Basado en problema matemático bien estudiado
✓ No requiere padding complicado
✓ Resistente a ataques de texto elegido

### Desventajas

✗ Cifrado es 2× el tamaño del mensaje
✗ Más lento que RSA
✗ Requiere generación de k aleatorio seguro
✗ Vulnerable si se reutiliza k

## 6. Comparación: ElGamal vs RSA

| Característica | ElGamal | RSA |
|---------------|---------|-----|
| Año | 1985 | 1977 |
| Base matemática | Logaritmo discreto | Factorización de primos |
| Tipo de cifrado | Probabilístico | Determinista |
| Expansión | 2× | ~1× |
| Velocidad | Más lento | Más rápido |
| Uso común | Firmas (DSA), PGP | SSL/TLS, firmas |
| Seguridad semántica | Sí (sin padding) | Requiere padding |

### Usos Actuales

**ElGamal:**
- Base del algoritmo DSA (Digital Signature Algorithm)
- PGP/GPG para cifrado de emails
- Algunas implementaciones de criptografía de curvas elípticas

**RSA:**
- SSL/TLS (HTTPS)
- Firma de certificados digitales
- SSH
- Más ampliamente usado en general

## 7. Ejemplo Detallado Paso a Paso

In [ ]:
print("="*100)
print(" "*30 + "EJEMPLO DETALLADO: Cifrar y Descifrar 'M' (12)")
print("="*100)

# Parámetros
p_ej = 23
g_ej = 5
x_ej = 6
h_ej = pow(g_ej, x_ej, p_ej)
m_ej = 12  # Letra 'M'
k_manual = 15  # Para este ejemplo, usamos k fijo

print(f"\n1. CONFIGURACIÓN")
print(f"   Clave pública:  (p={p_ej}, g={g_ej}, h={h_ej})")
print(f"   Clave privada:  x={x_ej}")
print(f"   Mensaje:        m={m_ej} (M)")
print(f"   k aleatorio:    k={k_manual}")

print(f"\n2. CIFRADO")
c1_manual = pow(g_ej, k_manual, p_ej)
print(f"   c1 = g^k mod p = {g_ej}^{k_manual} mod {p_ej} = {c1_manual}")

s_manual = pow(h_ej, k_manual, p_ej)
print(f"   s = h^k mod p = {h_ej}^{k_manual} mod {p_ej} = {s_manual}")

c2_manual = (m_ej * s_manual) % p_ej
print(f"   c2 = (m × s) mod p = ({m_ej} × {s_manual}) mod {p_ej} = {c2_manual}")

print(f"\n   🔒 Cifrado: ({c1_manual}, {c2_manual})")

print(f"\n3. DESCIFRADO")
s_desc = pow(c1_manual, x_ej, p_ej)
print(f"   s = c1^x mod p = {c1_manual}^{x_ej} mod {p_ej} = {s_desc}")
print(f"   ✓ Mismo s que en el cifrado: {s_manual == s_desc}")

s_inv_manual = pow(s_desc, p_ej - 2, p_ej)
print(f"\n   s^(-1) = s^(p-2) mod p = {s_desc}^{p_ej-2} mod {p_ej} = {s_inv_manual}")
print(f"   Verificación: ({s_desc} × {s_inv_manual}) mod {p_ej} = {(s_desc * s_inv_manual) % p_ej}")

m_desc = (c2_manual * s_inv_manual) % p_ej
print(f"\n   m = (c2 × s^(-1)) mod p = ({c2_manual} × {s_inv_manual}) mod {p_ej} = {m_desc}")

print(f"\n   🔓 Descifrado: {m_desc} (M)")
print(f"\n   ✅ Mensaje recuperado correctamente: {m_ej == m_desc}")
print("="*100)

---

## 📊 Suite de Pruebas Completa

In [ ]:
def suite_pruebas():
    """Suite completa de pruebas para validar la implementación."""
    
    print("="*100)
    print(" "*35 + "SUITE DE PRUEBAS")
    print("="*100)
    
    casos_prueba = [
        "HI",
        "HELLO",
        "CRYPTO",
        "ELGAMAL",
        "SECURITY"
    ]
    
    exitos = 0
    fallos = 0
    
    for i, mensaje in enumerate(casos_prueba, 1):
        print(f"\nPrueba {i}: {mensaje}")
        print("-"*100)
        
        try:
            # Cifrar
            cifrado = cifrar_texto(mensaje, p, g, h)
            
            # Descifrar
            descifrado, _ = descifrar_texto(cifrado, p, x)
            
            # Verificar
            correcto = (descifrado == mensaje)
            
            print(f"Original:   {mensaje}")
            print(f"Descifrado: {descifrado}")
            
            if correcto:
                print("✅ ÉXITO")
                exitos += 1
            else:
                print("❌ FALLO")
                fallos += 1
        
        except Exception as e:
            print(f"❌ ERROR: {e}")
            fallos += 1
    
    # Resumen
    print("\n" + "="*100)
    print(" "*40 + "RESUMEN")
    print("="*100)
    print(f"Total de pruebas: {len(casos_prueba)}")
    print(f"✅ Éxitos: {exitos}")
    print(f"❌ Fallos: {fallos}")
    print(f"Tasa de éxito: {(exitos/len(casos_prueba)*100):.1f}%")
    print("="*100)

# Ejecutar suite de pruebas
suite_pruebas()

---

## 🎓 Conclusiones y Referencias

### Lo que aprendimos

1. **ElGamal**: Sistema asimétrico basado en logaritmo discreto
2. **Cifrado Probabilístico**: Mayor seguridad que cifrados deterministas
3. **Claves Públicas/Privadas**: Fundamento de la criptografía moderna
4. **Grupos Multiplicativos**: Generadores y estructura algebraica

### Aplicaciones Prácticas

- **DSA**: Digital Signature Algorithm (basado en ElGamal)
- **PGP/GPG**: Cifrado de emails
- **Criptografía de Curvas Elípticas**: Variante moderna más eficiente

### Limitaciones de esta Implementación

⚠️ Esta es una implementación **educativa**:
- Usa primos pequeños (16 bits vs 2048+ bits en producción)
- No incluye padding ni codificación segura
- Generación de k no es criptográficamente segura
- Solo cifra letras (A-Z), no datos binarios arbitrarios

### Para Uso en Producción

Use bibliotecas establecidas como:
- **PyCryptodome**
- **cryptography** (Python)
- **OpenSSL**

### Referencias

- **ElGamal, T.** (1985): "A Public Key Cryptosystem and a Signature Scheme Based on Discrete Logarithms"
- **Katz & Lindell** (2014): "Introduction to Modern Cryptography"
- **Schneier, Bruce** (1996): "Applied Cryptography"
- **NIST FIPS 186-4**: Digital Signature Standard (DSS)

---

**Laboratorio completado exitosamente** ✅